# 3D Eye Tracker - Google Colab Version

This notebook provides a simplified version of the Orlosky 3D Eye Tracker that works in Google Colab.

**Features:**
- Real-time pupil detection and ellipse fitting
- 3D gaze vector computation
- Works with webcam or uploaded video
- Outputs gaze direction vectors

**Usage:**
1. Run all cells in order
2. Select input source (webcam or video upload)
3. View real-time eye tracking results

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install opencv-python numpy mediapipe -q
print("✓ Dependencies installed successfully")

## Step 2: Import Libraries and Setup

In [ ]:
import cv2
import numpy as np
import math
from IPython.display import display, Image, clear_output
from google.colab import files
from google.colab.patches import cv2_imshow
import time

print("✓ Libraries imported successfully")

## Step 3: Core Eye Tracking Functions

In [ ]:
# Crop image to maintain aspect ratio
def crop_to_aspect_ratio(image, width=640, height=480):
    current_height, current_width = image.shape[:2]
    desired_ratio = width / height
    current_ratio = current_width / current_height

    if current_ratio > desired_ratio:
        new_width = int(desired_ratio * current_height)
        offset = (current_width - new_width) // 2
        cropped_img = image[:, offset:offset+new_width]
    else:
        new_height = int(current_width / desired_ratio)
        offset = (current_height - new_height) // 2
        cropped_img = image[offset:offset+new_height, :]

    return cv2.resize(cropped_img, (width, height))

# Apply binary threshold
def apply_binary_threshold(image, darkestPixelValue, addedThreshold):
    threshold = darkestPixelValue + addedThreshold
    _, thresholded_image = cv2.threshold(image, threshold, 255, cv2.THRESH_BINARY_INV)
    return thresholded_image

# Find darkest area in image
def get_darkest_area(image):
    ignoreBounds = 20
    imageSkipSize = 10
    searchArea = 20
    internalSkipSize = 5
    
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    min_sum = float('inf')
    darkest_point = None

    for y in range(ignoreBounds, gray.shape[0] - ignoreBounds, imageSkipSize):
        for x in range(ignoreBounds, gray.shape[1] - ignoreBounds, imageSkipSize):
            current_sum = 0
            num_pixels = 0
            for dy in range(0, searchArea, internalSkipSize):
                if y + dy >= gray.shape[0]:
                    break
                for dx in range(0, searchArea, internalSkipSize):
                    if x + dx >= gray.shape[1]:
                        break
                    current_sum += gray[y + dy][x + dx]
                    num_pixels += 1

            if current_sum < min_sum and num_pixels > 0:
                min_sum = current_sum
                darkest_point = (x + searchArea // 2, y + searchArea // 2)

    return darkest_point

# Mask outside square
def mask_outside_square(image, center, size):
    x, y = center
    half_size = size // 2
    mask = np.zeros_like(image)
    top_left_x = max(0, x - half_size)
    top_left_y = max(0, y - half_size)
    bottom_right_x = min(image.shape[1], x + half_size)
    bottom_right_y = min(image.shape[0], y + half_size)
    mask[top_left_y:bottom_right_y, top_left_x:bottom_right_x] = 255
    return cv2.bitwise_and(image, mask)

# Filter contours by area and return largest
def filter_contours_by_area_and_return_largest(contours, pixel_thresh, ratio_thresh):
    max_area = 0
    largest_contour = None

    for contour in contours:
        area = cv2.contourArea(contour)
        if area >= pixel_thresh:
            x, y, w, h = cv2.boundingRect(contour)
            length_to_width_ratio = max(w / h, h / w)
            if length_to_width_ratio <= ratio_thresh:
                if area > max_area:
                    max_area = area
                    largest_contour = contour

    return [largest_contour] if largest_contour is not None else []

# Check ellipse goodness
def check_ellipse_goodness(binary_image, contour):
    if len(contour) < 5:
        return 0
    
    ellipse = cv2.fitEllipse(contour)
    mask = np.zeros_like(binary_image)
    cv2.ellipse(mask, ellipse, (255), -1)
    ellipse_area = np.sum(mask == 255)
    
    if ellipse_area == 0:
        return 0
    
    covered_pixels = np.sum((binary_image == 255) & (mask == 255))
    return covered_pixels / ellipse_area

# Process frame for pupil detection
def process_frame(frame):
    frame = crop_to_aspect_ratio(frame)
    darkest_point = get_darkest_area(frame)
    
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    darkest_pixel_value = gray_frame[darkest_point[1], darkest_point[0]]
    
    # Apply thresholding
    thresholded_image_strict = apply_binary_threshold(gray_frame, darkest_pixel_value, 5)
    thresholded_image_strict = mask_outside_square(thresholded_image_strict, darkest_point, 250)

    thresholded_image_medium = apply_binary_threshold(gray_frame, darkest_pixel_value, 15)
    thresholded_image_medium = mask_outside_square(thresholded_image_medium, darkest_point, 250)
    
    thresholded_image_relaxed = apply_binary_threshold(gray_frame, darkest_pixel_value, 25)
    thresholded_image_relaxed = mask_outside_square(thresholded_image_relaxed, darkest_point, 250)
    
    # Process thresholded images
    image_array = [thresholded_image_relaxed, thresholded_image_medium, thresholded_image_strict]
    kernel = np.ones((5, 5), np.uint8)
    
    best_ellipse = None
    best_goodness = 0
    
    for thresh_img in image_array:
        dilated = cv2.dilate(thresh_img, kernel, iterations=2)
        contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        reduced_contours = filter_contours_by_area_and_return_largest(contours, 1000, 3)
        
        if len(reduced_contours) > 0 and len(reduced_contours[0]) > 5:
            goodness = check_ellipse_goodness(dilated, reduced_contours[0])
            if goodness > best_goodness:
                best_goodness = goodness
                best_ellipse = cv2.fitEllipse(reduced_contours[0])
    
    # Draw results
    if best_ellipse is not None:
        cv2.ellipse(frame, best_ellipse, (0, 255, 0), 2)
        center_x, center_y = map(int, best_ellipse[0])
        cv2.circle(frame, (center_x, center_y), 3, (255, 255, 0), -1)
        
        # Display coordinates
        cv2.putText(frame, f"Pupil: ({center_x}, {center_y})", (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    
    return frame, best_ellipse

print("✓ Eye tracking functions loaded")

## Step 4: 3D Gaze Vector Computation

In [ ]:
def compute_gaze_vector(x, y, center_x=320, center_y=240, screen_width=640, screen_height=480):
    """
    Compute 3D gaze direction from pupil screen coordinates
    Returns: sphere_center, gaze_direction (both 3D vectors)
    """
    viewport_width = screen_width
    viewport_height = screen_height
    
    fov_y_deg = 45.0
    aspect_ratio = viewport_width / viewport_height
    far_clip = 100.0
    
    camera_position = np.array([0.0, 0.0, 3.0])
    
    fov_y_rad = np.radians(fov_y_deg)
    half_height_far = np.tan(fov_y_rad / 2) * far_clip
    half_width_far = half_height_far * aspect_ratio
    
    ndc_x = (2.0 * x) / viewport_width - 1.0
    ndc_y = 1.0 - (2.0 * y) / viewport_height
    
    far_x = ndc_x * half_width_far
    far_y = ndc_y * half_height_far
    far_z = camera_position[2] - far_clip
    far_point = np.array([far_x, far_y, far_z])
    
    ray_origin = camera_position
    ray_direction = far_point - camera_position
    ray_direction /= np.linalg.norm(ray_direction)
    ray_direction = -ray_direction
    
    inner_radius = 1.0 / 1.05
    sphere_offset_x = (center_x / screen_width) * 2.0 - 1.0
    sphere_offset_y = 1.0 - (center_y / screen_height) * 2.0
    sphere_center = np.array([sphere_offset_x * 1.5, sphere_offset_y * 1.5, 0.0])
    
    origin = ray_origin
    direction = -ray_direction
    L = origin - sphere_center
    
    a = np.dot(direction, direction)
    b = 2 * np.dot(direction, L)
    c = np.dot(L, L) - inner_radius**2
    
    discriminant = b**2 - 4 * a * c
    
    if discriminant < 0:
        return None, None
    
    sqrt_disc = np.sqrt(discriminant)
    t1 = (-b - sqrt_disc) / (2 * a)
    t2 = (-b + sqrt_disc) / (2 * a)
    
    t = None
    if t1 > 0 and t2 > 0:
        t = min(t1, t2)
    elif t1 > 0:
        t = t1
    elif t2 > 0:
        t = t2
    
    if t is None:
        return None, None
    
    intersection_point = origin + t * direction
    intersection_local = intersection_point - sphere_center
    target_direction = intersection_local / np.linalg.norm(intersection_local)
    
    return sphere_center, target_direction

print("✓ Gaze vector computation loaded")

## Step 5: Video Upload (Optional)

In [ ]:
# Upload a video file (optional)
print("Upload an eye tracking video (or skip to use test video):")
uploaded = files.upload()

if uploaded:
    video_path = list(uploaded.keys())[0]
    print(f"✓ Video uploaded: {video_path}")
else:
    video_path = None
    print("No video uploaded - will use default if available")

## Step 6: Run Eye Tracker

In [ ]:
def run_eye_tracker(video_path=None, max_frames=100):
    """
    Run eye tracker on video
    """
    if video_path is None:
        print("No video provided. Please upload a video first.")
        return
    
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print("Error: Could not open video")
        return
    
    frame_count = 0
    gaze_vectors = []
    
    print(f"Processing video... (max {max_frames} frames)")
    
    while frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        
        processed_frame, ellipse = process_frame(frame)
        
        if ellipse is not None:
            center_x, center_y = map(int, ellipse[0])
            sphere_center, gaze_direction = compute_gaze_vector(center_x, center_y)
            
            if sphere_center is not None and gaze_direction is not None:
                gaze_vectors.append({
                    'frame': frame_count,
                    'sphere_center': sphere_center,
                    'gaze_direction': gaze_direction
                })
                
                # Draw gaze info
                info_text = f"Gaze: ({gaze_direction[0]:.2f}, {gaze_direction[1]:.2f}, {gaze_direction[2]:.2f})"
                cv2.putText(processed_frame, info_text, (10, 60),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
        
        # Display frame every 10 frames
        if frame_count % 10 == 0:
            clear_output(wait=True)
            cv2_imshow(processed_frame)
            print(f"Processing frame {frame_count}/{max_frames}")
        
        frame_count += 1
    
    cap.release()
    
    print(f"\n✓ Processing complete! Processed {frame_count} frames")
    print(f"✓ Detected gaze in {len(gaze_vectors)} frames")
    
    return gaze_vectors

# Run tracker if video is available
if video_path:
    gaze_data = run_eye_tracker(video_path, max_frames=100)
else:
    print("Please upload a video in the previous cell first")

## Step 7: Export Results

In [ ]:
# Export gaze vectors to CSV
if 'gaze_data' in locals() and gaze_data:
    import csv
    
    with open('gaze_vectors.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Frame', 'Sphere_X', 'Sphere_Y', 'Sphere_Z', 
                        'Gaze_X', 'Gaze_Y', 'Gaze_Z'])
        
        for data in gaze_data:
            writer.writerow([
                data['frame'],
                data['sphere_center'][0],
                data['sphere_center'][1],
                data['sphere_center'][2],
                data['gaze_direction'][0],
                data['gaze_direction'][1],
                data['gaze_direction'][2]
            ])
    
    print("✓ Gaze vectors exported to gaze_vectors.csv")
    files.download('gaze_vectors.csv')
else:
    print("No gaze data available. Run the tracker first.")

## Testing Instructions

To test the eye tracker:

1. **Upload a video**: Use the upload cell above to select an eye tracking video
   - Video should show a close-up of an eye with good lighting
   - IR camera videos work best
   - Standard webcam videos will also work

2. **Run the tracker**: Execute the "Run Eye Tracker" cell
   - Processed frames will be displayed
   - Green ellipse shows detected pupil
   - Yellow dot shows pupil center
   - Gaze vector is displayed at bottom

3. **Export results**: Run the export cell to download gaze vectors as CSV

4. **Test multiple times**: Run the tracker cell multiple times with different videos to verify robustness

### Sample Test Videos
You can use the included `eye_test.mp4` from the repository, or record your own eye video.

### Expected Results
- Green ellipse should accurately track the pupil
- Gaze vector should update smoothly
- CSV export should contain all detected frames
